# StemScore 分離モデルの取得

StemScore の「6パート」「高品質」モデルを、公式の配布元からダウンロードして Google ドライブに保存します。
(PC のネットワークから配布元に接続できない場合用)

1. 下のセルの ▶ を押して実行（Google アカウントへのアクセス許可を求められたら許可）
2. 数分後、最後に表示されるリンクをすべてコピー
3. StemScore の「分離モデルの追加」に貼り付けて「モデルを取り込む」

ドライブには `StemScore_models` フォルダが作られます。取り込みが終わったら削除して構いません。

In [ ]:
# StemScore 用の分離モデル (Demucs, MIT ライセンス) を公式配布元から取得して Google ドライブへ保存
FILES = [
    "5c90dfd2-34c22ccb.th",  # 6パート (htdemucs_6s)
    "f7e0c4bc-ba3fe64a.th", "d12395a8-e57c48e6.th",
    "92cfc3b6-ef3bcb9c.th", "04573f0d-f3cf25b2.th",  # 高品質 (htdemucs_ft)
]
BASE = "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/"

import hashlib, urllib.request
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()
drive = build("drive", "v3")
folder = drive.files().create(
    body={"name": "StemScore_models", "mimeType": "application/vnd.google-apps.folder"},
    fields="id").execute()["id"]

links = []
for f in FILES:
    path = "/content/" + f
    print("ダウンロード中:", f)
    urllib.request.urlretrieve(BASE + f, path)
    h = hashlib.sha256(open(path, "rb").read()).hexdigest()
    assert h.startswith(f.split("-")[1][:8]), "ファイルが壊れています: " + f
    up = drive.files().create(body={"name": f, "parents": [folder]},
                              media_body=MediaFileUpload(path, resumable=True),
                              fields="id").execute()
    # PC 側の StemScore がダウンロードできるよう「リンクを知っている全員」に共有 (公開モデルのため問題なし)
    drive.permissions().create(fileId=up["id"], body={"type": "anyone", "role": "reader"}).execute()
    links.append(f"https://drive.google.com/file/d/{up['id']}/view")

print("\n↓ 以下のリンクをすべてコピーして StemScore の「分離モデルの追加」に貼り付けてください\n")
print("\n".join(links))
